# MultiFC Label-Granularity Experiments — Native 3-Class → 2-Class Collapse

**Purpose.** Run the *identical* chronological, restricted-metadata **DA-MiniLM-RWAC**
protocol used on LIAR2, but on the **MultiFC** external-replication dataset, at two
label granularities: the native harmonised **3 classes** `{false, mixed, true}` and a
derived **2 classes** `{false, true}`. At each granularity we compare DA-MiniLM-RWAC
against **MiniLM-LogReg-RW** (the primary confirmatory comparator) and **Uniform-LR**
(the unweighted frozen-embedding baseline).

**Why this notebook exists.** The LIAR2 collapse notebook imports four project classes
(`Liar2Preprocessor`, `TemporalSplitter`, `DAMiniLMModel`, `DAMiniLMExperimentRunner`)
and adds a label-collapse layer. This notebook is the **MultiFC port**: it is fully
self-contained (all four classes are defined inline, with `Liar2Preprocessor` replaced
by a new **`MultiFCPreprocessor`**), so it runs on Kaggle by only editing the three
MultiFC TSV paths in Section 0. **No model, weighting, calibration, or evaluation logic
is changed** — only the data-loading front end differs, which is exactly what the
"independent retraining on an external dataset" framing requires.

**Framing (must stay consistent across the manuscript).** MultiFC here is an
**external-dataset replication with independent retraining**, *not* zero-shot transfer
and *not* a generalization claim. The complete DA-MiniLM-RWAC procedure is retrained
from scratch on MultiFC embeddings; nothing is carried over from the LIAR2 fit.

**Research question.** Does DA-MiniLM-RWAC's LIAR2 advantage over MiniLM-LogReg-RW
reproduce on MultiFC? A *non-reproducing* advantage — the simpler comparator remaining
equally or more competitive — is itself the reportable external-replication result.

## Methodological invariants (unchanged from the LIAR2 pipeline)

1. **Collapse happens *after* the clean / chronological-order / leakage-drop /
   `time_rank` pipeline.** Only `label_id` is remapped, so the 2-class run sits on the
   *exact same rows* as the 3-class run — a clean, paired comparison. (Rows whose native
   label is `mixed` are folded into `false` for the 2-class scheme; no rows are dropped
   at collapse time.)
2. **Macro-F1 stays the primary metric.** MultiFC is severely class-imbalanced, so
   accuracy inflation is *worse* here, making macro-F1 even more necessary.
3. **Class-balance sample weights are recomputed per granularity** (fresh model +
   runner with the correct `num_classes`); weights are never reused across K.
4. **Embeddings are computed once and reused across granularities** — collapsing labels
   does not change model inputs, so re-encoding would be wasted compute.
5. **Restricted-metadata text mode is the headline condition.** For MultiFC the
   restricted metadata is `{category, speaker, checker, tags, entities}`; the
   fact-check *verdict/evidence* text is **excluded** as a leakage control (it often
   states the answer).

## How to run on Kaggle

1. Enable **Internet** (Settings → Internet → On) — required for MiniLM embeddings.
2. Enable a **GPU** accelerator (recommended for the MLP head).
3. Attach the MultiFC dataset and update the three TSV paths in **Section 0**.
4. Run all cells top to bottom. Nothing needs to be pasted — the four classes are inline.

> **Note on the MultiFC test split.** MultiFC's official `test.tsv` ships *without gold
> labels* (12 columns instead of 13), so it cannot be scored. As in the LIAR2 protocol,
> evaluation is done on an **internal chronological split** of the *labelled* rows
> (`train.tsv` + `dev.tsv` pooled, ordered by date). The unlabelled official test rows
> are ignored.


## Section 0 — Configuration

In [4]:
# ============================================================
# 0a. Install dependencies if needed
# ============================================================
import sys, subprocess, importlib

_required_packages = {
    "sentence_transformers": "sentence-transformers",
    "torch": "torch",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "numpy": "numpy",
    "scipy": "scipy",
}
for _imp, _pip in _required_packages.items():
    try:
        importlib.import_module(_imp)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pip])
        print(f"installed {_pip}")
print("Dependencies ready.")


Dependencies ready.


In [5]:
# ============================================================
# 0b. Imports and experiment configuration
# ============================================================
import os, re, json, math, random, warnings, glob
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sentence_transformers import SentenceTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score,
    recall_score, confusion_matrix, log_loss,
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore")

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ----- EDIT THESE THREE PATHS to match your Kaggle MultiFC dataset -----
# The loader also auto-searches common Kaggle input locations if these are wrong.
MULTIFC_TRAIN = "/kaggle/input/datasets/badalahmmed/multifc/train.tsv"
MULTIFC_DEV   = "/kaggle/input/datasets/badalahmmed/multifc/dev.tsv"
MULTIFC_TEST  = "/kaggle/input/datasets/badalahmmed/multifc/test.tsv"   # unlabelled; used only for row-count reporting

OUTPUT_DIR = "/kaggle/working/multifc_label_collapse_results"
if not Path("/kaggle").exists():
    OUTPUT_DIR = "/mnt/data/multifc_label_collapse_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----- Experiment configuration (mirrors the LIAR2 collapse notebook) -----
# Restricted-metadata text mode is the paper's headline condition.
# (In-code the column is named "claim_plus_safe_metadata"; the paper calls it
#  "restricted metadata" — same thing.)
TEXT_MODE   = "claim_plus_safe_metadata"
SPLIT_MODE  = "chronological"          # headline split
SEEDS       = [0, 1, 2, 3, 4]          # 5-seed battery
N_BOOT      = 2000                     # paired-bootstrap resamples

# MultiFC has no 6-class ordinal labels. Native granularity is 3-class; 2-class is
# derived from it (mixed -> false). The 6-class scheme is intentionally absent.
SCHEMES = ["3class", "2class"]

# Set True to also run the plain claim-only text mode at each granularity.
ALSO_RUN_CLAIM_ONLY = False

# MiniLM / MLP hyperparameters (identical to the LIAR2 main model)
MINILM_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_BATCH_SIZE  = 128
MLP_EPOCHS        = 35
MLP_BATCH_SIZE    = 128
MLP_LR            = 1e-3
MLP_WEIGHT_DECAY  = 1e-4
MLP_PATIENCE      = 6
RECENCY_LAMBDAS   = [0.0, 0.05, 0.10, 0.20, 0.30]
CLASS_WEIGHT_BETA = 0.999
TEMPERATURE_GRID  = [0.75, 1.0, 1.25, 1.5, 2.0]
TAU_GRID          = [0.0, 0.5, 1.0, 1.5, 2.0]

print("Config loaded. Output dir:", OUTPUT_DIR)
print("Schemes:", SCHEMES, "| headline text mode:", TEXT_MODE, "| split:", SPLIT_MODE)


Using device: cuda
Config loaded. Output dir: /kaggle/working/multifc_label_collapse_results
Schemes: ['3class', '2class'] | headline text mode: claim_plus_safe_metadata | split: chronological


## Section 1 — Project classes (defined inline)

The four classes the LIAR2 collapse notebook expects are defined below. Three of them
(`TemporalSplitter`, `DAMiniLMModel`, `DAMiniLMExperimentRunner`) are **dataset-agnostic
and identical** to the LIAR2 project versions — they operate on embeddings + `label_id`
+ `time_rank`, none of which is LIAR2-specific. The fourth, `Liar2Preprocessor`, is
replaced by **`MultiFCPreprocessor`**, which is the only genuinely MultiFC-specific
component.


### 1a. `MultiFCPreprocessor`

Loads MultiFC's tab-separated splits, harmonises the 26-site verdict taxonomy down to
`{false, mixed, true}`, cleans text and dates, builds the `claim_only` and
`claim_plus_safe_metadata` (restricted-metadata) text columns, orders rows
chronologically, and attaches a normalised `time_rank`. The public surface deliberately
mirrors `Liar2Preprocessor` so the downstream runner does not know which dataset it is
looking at:

- `run_pipeline(train, dev, test)` → cleaned, chronologically-ordered `DataFrame`
- `audit_duplicate_statements(df)` → duplicate-claim audit table
- `drop_duplicate_statements(df, policy, audit_df)` → de-duplicated frame
- `parse_and_sort_by_date(df)` → re-sort by parsed date
- `add_time_rank(df)` → attach `time_rank` in `[0, 1]`

The label-harmonisation dictionaries are the same ones used in the standalone MultiFC
experiments; `label_harmonization_report()` writes the full raw→3-class mapping audit
required by the manuscript's *MultiFC label-harmonization documentation*.


In [6]:
# ============================================================
# 1a. MultiFCPreprocessor  (replaces Liar2Preprocessor)
# ============================================================

# 3-class harmonised taxonomy — identical to the collapsed LIAR2 taxonomy, so the
# two datasets' 3-class numbers are directly comparable.
MULTIFC_LABEL_ORDER = ["false", "mixed", "true"]
MULTIFC_LABEL_TO_ID = {l: i for i, l in enumerate(MULTIFC_LABEL_ORDER)}
MULTIFC_ID_TO_LABEL = {i: l for l, i in MULTIFC_LABEL_TO_ID.items()}


def _mfc_norm_label(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    x = str(x).strip().lower().replace("_", "-")
    return re.sub(r"\s+", " ", x)


# Verdict strings pooled across the 26 MultiFC fact-checking sites, bucketed into three
# ordinal classes. Anything not listed (including bare numeric ratings whose
# site-specific meaning is unverified) is dropped as unmappable.
_MFC_TRUE = set(_mfc_norm_label(x) for x in [
    "true", "truth!", "verdict: true", "determination: true",
    "factscan score: true", "correct", "accurate", "verified", "fact",
    "conclusion: accurate", "mostly true", "mostly-true", "mostly truth!",
    "mostly-correct", "determination: mostly true", "promise kept",
    "in-the-green", "true messages", "authorship confirmed!",
    "confirmed authorship!", "correct attribution", "correct attribution!",
    "truth! & outdated!", "truth! & unproven!", "previously truth! now resolved!",
])
_MFC_MIXED = set(_mfc_norm_label(x) for x in [
    "half-true", "half true", "mixture", "in-between", "partly true",
    "partially true", "needs context", "not the whole story", "compromise",
    "determination: barely true", "truth! & fiction!", "truth! & misleading!",
    "truth! & disputed!", "disputed!", "unproven", "unproven!", "unverified",
    "determination: misleading", "factscan score: misleading", "misleading",
    "misleading!", "misleading recommendations", "2 pinnochios",
    "a little baloney", "some baloney", "spins the facts", "distorts the facts",
    "cherry picks", "exaggerated", "exaggerates", "understated", "half flip",
    "no flip", "conclusion: unclear", "determination: a stretch",
])
_MFC_FALSE = set(_mfc_norm_label(x) for x in [
    "false", "mostly false", "mostly-false", "pants on fire!", "pants-fire",
    "fiction!", "fiction", "mostly fiction!", "verdict: false",
    "determination: false", "factscan score: false", "conclusion: false",
    "rating: false", "we rate this claim false", "incorrect",
    "incorrect attribution!", "inaccurate attribution!", "fake", "fake news",
    "facebook scams", "scam", "scam!", "bogus warning", "virus!",
    "4 pinnochios", "3 pinnochios", "a lot of baloney", "full flop",
    "promise broken", "in-the-red", "outdated", "outdated!", "misattributed",
    "miscaptioned", "unsupported", "unsubstantiated messages",
    "verdict: unsubstantiated", "no evidence",
])
_MFC_DROP = set(_mfc_norm_label(x) for x in [
    "none", "not yet rated", "in the works", "stalled",
    "investigation pending!", "opinion!", "commentary!", "unobservable",
    "grass roots movement!", "legend", "fiction! & satire!",
    "statirical reports", "determination: huckster propaganda",
])


class MultiFCPreprocessor:
    """MultiFC front-end mirroring the Liar2Preprocessor public surface.

    The 13-column labelled MultiFC schema is:
        claim_id, claim, raw_label, url, evidence_text, category, speaker,
        checker, tags, title, published_date, claim_date, entities
    The 12-column unlabelled schema drops `raw_label`.

    Restricted-metadata text mode uses ONLY non-leaky metadata
    ({category, speaker, checker, tags, entities}); the fact-check verdict / evidence /
    title / url are never concatenated into the model text.
    """

    LABELLED_COLUMNS = [
        "claim_id", "claim", "raw_label", "url", "evidence_text", "category",
        "speaker", "checker", "tags", "title", "published_date", "claim_date",
        "entities",
    ]
    UNLABELLED_COLUMNS = [
        "claim_id", "claim", "url", "evidence_text", "category", "speaker",
        "checker", "tags", "title", "published_date", "claim_date", "entities",
    ]
    # Non-leaky metadata folded into the restricted-metadata text mode.
    SAFE_METADATA_COLUMNS = ["category", "speaker", "checker", "tags", "entities"]
    TEXT_MODES = ["claim_only", "claim_plus_safe_metadata"]

    def __init__(self, output_dir=None, min_claim_chars=1):
        self.output_dir = output_dir
        self.min_claim_chars = min_claim_chars
        self.label_order = MULTIFC_LABEL_ORDER
        self.label_to_id = MULTIFC_LABEL_TO_ID
        self._last_label_report = None

    # ---------- low-level cleaning ----------
    @staticmethod
    def _clean_text(x):
        if x is None or (isinstance(x, float) and pd.isna(x)):
            return ""
        x = str(x).replace("\n", " ").replace("\r", " ")
        x = re.sub(r"\s+", " ", x).strip()
        if x.lower() == "none":
            return ""
        return x

    @staticmethod
    def _clean_entities(x):
        """MultiFC `entities` is a stringified list like "['A', 'B']"; flatten to
        a readable space-joined string and drop the 'None' placeholder."""
        s = MultiFCPreprocessor._clean_text(x)
        if not s:
            return ""
        found = re.findall(r"'([^']*)'", s)
        if found:
            found = [f for f in found if f and f.lower() != "none"]
            return ", ".join(found)
        return "" if s.lower() in {"['none']", "[none]"} else s

    @classmethod
    def _map_label(cls, raw_label):
        x = _mfc_norm_label(raw_label)
        if x in _MFC_TRUE:  return "true"
        if x in _MFC_MIXED: return "mixed"
        if x in _MFC_FALSE: return "false"
        if x in _MFC_DROP:  return np.nan
        if re.fullmatch(r"\d+", x):  # bare numeric ratings: meaning unverified -> drop
            return np.nan
        return np.nan

    @staticmethod
    def _parse_date(row):
        d1 = pd.to_datetime(row.get("published_date", ""), errors="coerce", utc=True)
        if pd.notna(d1):
            return d1
        d2 = pd.to_datetime(row.get("claim_date", ""), errors="coerce", utc=True)
        if pd.notna(d2):
            return d2
        return pd.NaT

    # ---------- raw TSV loading ----------
    def _read_tsv(self, path, split_name):
        raw = pd.read_csv(path, sep="\t", header=None, dtype=str,
                          keep_default_na=False, quoting=3, engine="python")
        if raw.shape[1] == 13:
            raw.columns = self.LABELLED_COLUMNS
            raw["has_label"] = True
        elif raw.shape[1] == 12:
            raw.columns = self.UNLABELLED_COLUMNS
            raw["raw_label"] = np.nan
            raw["has_label"] = False
        else:
            raise ValueError(
                f"{path} has {raw.shape[1]} columns; expected 13 (labelled) or 12 "
                f"(unlabelled)."
            )
        raw["split"] = split_name
        raw["source_site"] = (
            raw["claim_id"].astype(str).str.extract(r"^([A-Za-z]+)", expand=False)
            .fillna("unknown")
        )
        return raw

    def _standardize(self, df):
        out = df.copy()
        for col in ["claim", "evidence_text", "category", "speaker", "checker",
                    "tags", "title", "source_site"]:
            if col in out.columns:
                out[col] = out[col].apply(self._clean_text)
        out["entities"] = out["entities"].apply(self._clean_entities) \
            if "entities" in out.columns else ""
        out["date_parsed"] = out.apply(self._parse_date, axis=1)
        out["label_3class"] = out["raw_label"].apply(self._map_label)
        return out

    def _build_text(self, row, text_mode):
        claim = self._clean_text(row.get("claim", ""))
        if text_mode == "claim_only":
            return f"claim: {claim}"
        if text_mode == "claim_plus_safe_metadata":
            parts = [f"claim: {claim}"]
            for col in self.SAFE_METADATA_COLUMNS:
                val = self._clean_text(row.get(col, ""))
                if val:
                    parts.append(f"{col}: {val}")
            return " [SEP] ".join(parts)
        raise ValueError(f"Unknown text_mode: {text_mode}")

    # ---------- public API (mirrors Liar2Preprocessor) ----------
    def run_pipeline(self, train_path, dev_path, test_path=None):
        """Load + harmonise + clean + build text columns + chronological order.

        Only *labelled* rows are retained (official unlabelled test rows are dropped,
        since they cannot be scored). Rows with an unmappable verdict or an empty claim
        are dropped. Returns a frame carrying `label_id`, `time_rank`, and both
        `model_text__*` columns.
        """
        frames = [self._read_tsv(train_path, "train"),
                  self._read_tsv(dev_path, "dev")]
        n_official_test = 0
        if test_path:
            try:
                _te = self._read_tsv(test_path, "test")
                n_official_test = len(_te)
                # kept only for reporting; unlabelled rows are not used downstream
            except Exception as e:
                print("NOTE: could not read official test.tsv ->", repr(e))

        std = pd.concat([self._standardize(f) for f in frames], ignore_index=True)

        # label-harmonisation audit (before dropping) -> manuscript documentation
        self._last_label_report = (
            std.groupby(["raw_label", "label_3class"], dropna=False)
               .size().reset_index(name="count")
               .sort_values("count", ascending=False)
        )
        n_before = len(std)
        std = std[std["has_label"]].copy()
        std = std[std["label_3class"].isin(self.label_order)].copy()
        std["claim"] = std["claim"].apply(self._clean_text)
        std = std[std["claim"].str.len() >= self.min_claim_chars].copy()
        n_after = len(std)
        dropped_pct = 100.0 * (n_before - n_after) / max(1, n_before)

        std["label_id"] = std["label_3class"].map(self.label_to_id).astype(int)
        for mode in self.TEXT_MODES:
            std[f"model_text__{mode}"] = std.apply(
                lambda r: self._build_text(r, mode), axis=1)

        std = self.parse_and_sort_by_date(std)
        std["claim_uid"] = np.arange(len(std))
        std = self.add_time_rank(std)
        std = std.reset_index(drop=True)

        print(f"MultiFC pipeline complete.")
        print(f"  labelled rows kept : {n_after}  (dropped {n_before - n_after} "
              f"unmappable/empty of {n_before}, {dropped_pct:.1f}%)")
        if n_official_test:
            print(f"  official UNLABELLED test rows ignored (not scorable): "
                  f"{n_official_test}")
        print(f"  3-class distribution: "
              f"{std['label_3class'].value_counts().to_dict()}")
        return std

    def audit_duplicate_statements(self, df, text_col="claim"):
        """Return a table of claim strings that appear more than once."""
        key = df[text_col].astype(str).str.strip().str.lower()
        vc = key.value_counts()
        dup_keys = vc[vc > 1]
        audit = (pd.DataFrame({"claim_norm": dup_keys.index, "n_copies": dup_keys.values})
                 .reset_index(drop=True))
        print(f"Duplicate-claim audit: {len(audit)} claim strings occur >1x "
              f"({int((vc[vc > 1] - 1).sum())} redundant rows).")
        return audit

    def drop_duplicate_statements(self, df, policy="keep_first", audit_df=None,
                                  text_col="claim"):
        """De-duplicate on normalised claim text. policy='keep_first' keeps the
        earliest row (frame is date-sorted by run_pipeline before this is called)."""
        if policy != "keep_first":
            raise ValueError("only policy='keep_first' is supported")
        key = df[text_col].astype(str).str.strip().str.lower()
        keep_mask = ~key.duplicated(keep="first")
        out = df[keep_mask].copy()
        print(f"Dropped {len(df) - len(out)} duplicate-claim rows "
              f"(policy=keep_first). Remaining: {len(out)}.")
        return out

    def parse_and_sort_by_date(self, df):
        """Sort chronologically by parsed date. Rows with an unparseable date are
        placed last but keep their relative order (stable sort)."""
        out = df.copy()
        if "date_parsed" not in out.columns:
            out["date_parsed"] = out.apply(self._parse_date, axis=1)
        out["_no_date"] = out["date_parsed"].isna().astype(int)
        out = out.sort_values(["_no_date", "date_parsed"], kind="mergesort")
        out = out.drop(columns="_no_date").reset_index(drop=True)
        return out

    def add_time_rank(self, df):
        """Attach a strictly-increasing normalised time_rank in [0, 1] over the
        current row order (which run_pipeline guarantees is chronological)."""
        out = df.copy()
        n = len(out)
        out["time_rank"] = np.linspace(0.0, 1.0, n) if n > 1 else np.zeros(n)
        return out

    def label_harmonization_report(self, save_path=None):
        """Return (and optionally save) the raw->3-class mapping audit from the last
        run_pipeline call — the documentation the manuscript requires."""
        if self._last_label_report is None:
            raise RuntimeError("call run_pipeline first")
        rep = self._last_label_report.copy()
        if save_path:
            rep.to_csv(save_path, index=False)
            print("Saved label-harmonisation report:", save_path)
        return rep


print("MultiFCPreprocessor defined.")


MultiFCPreprocessor defined.


### 1b. Label-collapse layer

`LabelCollapser` remaps the native 3-class `label_id` to a coarser granularity. Only two
schemes exist for MultiFC: the identity 3-class scheme, and a 2-class scheme that folds
`mixed → false` (placing the ambiguous middle on the *false* side, the conservative
choice for a veracity screen). The original 3-class labels are retained as
`label_id_3class` for auditing.


In [7]:
# ============================================================
# 1b. Label-collapse definitions
# ============================================================
# 3-class identity (native MultiFC granularity)
THREECLASS_IDENTITY = {0: 0, 1: 1, 2: 2}
THREECLASS_ORDER    = ["false", "mixed", "true"]

# 2-class: {false, true}. mixed -> false (conservative veracity screen).
THREECLASS_TO_TWO   = {0: 0, 1: 0, 2: 1}
TWOCLASS_ORDER      = ["false", "true"]

COLLAPSE_SCHEMES = {
    "3class": {"map": THREECLASS_IDENTITY, "order": THREECLASS_ORDER, "num_classes": 3},
    "2class": {"map": THREECLASS_TO_TWO,   "order": TWOCLASS_ORDER,   "num_classes": 2},
}


class LabelCollapser:
    """Remaps an already-cleaned MultiFC DataFrame's `label_id` to a coarser
    granularity, leaving claim text, date order, time_rank and the restricted-metadata
    text modes untouched. Native 3-class labels are retained as `label_id_3class`."""

    def __init__(self, scheme: str):
        if scheme not in COLLAPSE_SCHEMES:
            raise ValueError(f"scheme must be one of {list(COLLAPSE_SCHEMES)}, "
                             f"got {scheme!r}")
        cfg = COLLAPSE_SCHEMES[scheme]
        self.scheme = scheme
        self.map = cfg["map"]
        self.label_order = cfg["order"]
        self.num_classes = cfg["num_classes"]

    def transform(self, df: pd.DataFrame, label_id_col: str = "label_id") -> pd.DataFrame:
        if label_id_col not in df.columns:
            raise KeyError(f"{label_id_col!r} not in DataFrame columns")
        out = df.copy()
        three = out[label_id_col].astype(int)
        unknown = set(three.unique()) - set(self.map)
        if unknown:
            raise ValueError(f"label_id values {sorted(unknown)} absent from "
                             f"{self.scheme} map (expected 0..2)")
        out["label_id_3class"] = three.values
        out[label_id_col] = three.map(self.map).astype(int).values
        out["label_collapsed"] = out[label_id_col].map(
            {i: n for i, n in enumerate(self.label_order)}).values
        return out

    def describe_collapse(self, df: pd.DataFrame,
                          label_id_col: str = "label_id") -> pd.DataFrame:
        three_col = "label_id_3class" if "label_id_3class" in df.columns else label_id_col
        rows = []
        for sid in range(3):
            n = int((df[three_col] == sid).sum())
            cid = self.map[sid]
            rows.append({"three_id": sid, "three_label": THREECLASS_ORDER[sid],
                         "collapsed_id": cid,
                         "collapsed_label": self.label_order[cid], "n_rows": n})
        tbl = pd.DataFrame(rows)
        totals = (tbl.groupby(["collapsed_id", "collapsed_label"])["n_rows"].sum()
                     .reset_index().rename(columns={"n_rows": "collapsed_total"}))
        return tbl.merge(totals, on=["collapsed_id", "collapsed_label"])


# quick self-test on synthetic ids
_demo = pd.DataFrame({"label_id": [0, 1, 2]})
for _s in COLLAPSE_SCHEMES:
    _c = LabelCollapser(_s).transform(_demo)
    print(f"{_s:8s} -> {_c['label_id'].tolist()}  (K={LabelCollapser(_s).num_classes})")


3class   -> [0, 1, 2]  (K=3)
2class   -> [0, 0, 1]  (K=2)


### 1c. `TemporalSplitter`

Dataset-agnostic 70/15/15 splitter. `chronological` slices the (already date-ordered)
frame into contiguous train/valid/test blocks; `random_stratified` draws a stratified
random split. The chronological test partition is identical across seeds by
construction.


In [8]:
# ============================================================
# 1c. TemporalSplitter  (dataset-agnostic; identical to LIAR2 project)
# ============================================================
from sklearn.model_selection import train_test_split


def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class TemporalSplitter:
    def __init__(self, train_ratio=0.70, valid_ratio=0.15, test_ratio=0.15):
        s = train_ratio + valid_ratio + test_ratio
        assert abs(s - 1.0) < 1e-6, f"ratios must sum to 1, got {s}"
        self.train_ratio = train_ratio
        self.valid_ratio = valid_ratio
        self.test_ratio = test_ratio

    def _chronological(self, df):
        n = len(df)
        tr_end = int(n * self.train_ratio)
        va_end = int(n * (self.train_ratio + self.valid_ratio))
        return (np.arange(0, tr_end), np.arange(tr_end, va_end), np.arange(va_end, n))

    def _random_stratified(self, df, seed=42):
        idx = np.arange(len(df))
        y = df["label_id"].values
        tr, tmp = train_test_split(idx, train_size=self.train_ratio,
                                   random_state=seed, stratify=y)
        rel = self.valid_ratio / (self.valid_ratio + self.test_ratio)
        va, te = train_test_split(tmp, train_size=rel, random_state=seed,
                                  stratify=y[tmp])
        return np.sort(tr), np.sort(va), np.sort(te)

    def get_split(self, df, split_mode, seed=42):
        if split_mode == "chronological":
            return self._chronological(df)
        if split_mode == "random_stratified":
            return self._random_stratified(df, seed=seed)
        raise ValueError(f"Unknown split_mode: {split_mode}")


### 1d. `DAMiniLMModel`

Owns the frozen-MiniLM encoder (with on-disk + in-memory embedding cache), the
DA-MiniLM MLP head, the MiniLM-LogReg estimator, the effective-number class weights,
exponential recency weights, the two-stage logit-adjustment + temperature calibration,
and all metric helpers. `num_classes` is set per granularity so the MLP head, Brier, and
NLL are sized correctly.


In [9]:
# ============================================================
# 1d. DAMiniLMModel  (dataset-agnostic; identical logic to LIAR2 main model)
# ============================================================
class _EmbeddingDataset(Dataset):
    def __init__(self, X, y=None, sample_weight=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)
        if sample_weight is None:
            sample_weight = np.ones(len(X), dtype=np.float32)
        self.sample_weight = torch.tensor(sample_weight, dtype=torch.float32)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        item = {"x": self.X[i], "sample_weight": self.sample_weight[i]}
        if self.y is not None:
            item["y"] = self.y[i]
        return item


class _DAMiniLMClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=256, dropout=0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes),
        )

    def forward(self, x):
        return self.net(x)


class DAMiniLMModel:
    """Frozen-MiniLM encoder + DA-MiniLM MLP head + LogReg estimator + weighting +
    two-stage calibration + metrics. One instance per class-granularity."""

    def __init__(self, output_dir, num_classes,
                 minilm_model_name=MINILM_MODEL_NAME,
                 device=DEVICE):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.num_classes = int(num_classes)
        self.minilm_model_name = minilm_model_name
        self.device = device
        self._embedding_cache = {}          # shared across instances by the runner
        self._sentence_model = None
        # calibration / weighting grids
        self.class_weight_beta = CLASS_WEIGHT_BETA
        self.tau_grid = TAU_GRID
        self.temperature_grid = TEMPERATURE_GRID

    # ---------- encoding ----------
    def encode_texts_minilm(self, texts, cache_key, batch_size=EMBED_BATCH_SIZE):
        cache_path = self.output_dir / f"embeddings_{cache_key}.npy"
        if cache_key in self._embedding_cache:
            return self._embedding_cache[cache_key]
        if cache_path.exists():
            X = np.load(cache_path)
            self._embedding_cache[cache_key] = X
            print(f"Loaded cached embeddings: {cache_path} {X.shape}")
            return X
        if self._sentence_model is None:
            self._sentence_model = SentenceTransformer(self.minilm_model_name,
                                                       device=str(self.device))
        X = self._sentence_model.encode(
            list(texts), batch_size=batch_size, show_progress_bar=True,
            convert_to_numpy=True, normalize_embeddings=True,
        ).astype(np.float32)
        np.save(cache_path, X)
        self._embedding_cache[cache_key] = X
        print(f"Saved embeddings: {cache_path} {X.shape}")
        return X

    # ---------- weighting ----------
    @staticmethod
    def _normalize_weights(w, min_w=0.05, max_w=20.0):
        w = np.clip(np.asarray(w, float), min_w, max_w)
        return w / (w.mean() + 1e-12)

    def _effective_number_class_weights(self, y):
        counts = np.bincount(np.asarray(y, int),
                             minlength=self.num_classes).astype(float)
        w = np.ones(self.num_classes, float)
        for c in range(self.num_classes):
            n = counts[c]
            if n > 0:
                eff = (1.0 - self.class_weight_beta ** n) / (1.0 - self.class_weight_beta)
                w[c] = 1.0 / eff
        return w / (w.mean() + 1e-12)

    def class_sample_weights(self, y_train):
        cw = self._effective_number_class_weights(y_train)
        return self._normalize_weights(cw[np.asarray(y_train, int)])

    def recency_sample_weights(self, time_rank_train, lam=0.0):
        t = np.asarray(time_rank_train, float)
        age = t.max() - t
        return self._normalize_weights(np.exp(-float(lam) * age * 10.0))

    def combined_sample_weights(self, y_train, time_rank_train, recency_lambda=0.0):
        return self._normalize_weights(
            self.class_sample_weights(y_train)
            * self.recency_sample_weights(time_rank_train, recency_lambda))

    # ---------- calibration primitives ----------
    @staticmethod
    def _softmax(logits):
        logits = np.asarray(logits, float)
        logits = logits - logits.max(axis=1, keepdims=True)
        e = np.exp(logits)
        return e / (e.sum(axis=1, keepdims=True) + 1e-12)

    @staticmethod
    def _proba_to_logits(p, eps=1e-12):
        return np.log(np.clip(p, eps, 1.0))

    def logit_adjustment_bias(self, y_train, tau=1.0, eps=1e-12):
        counts = np.bincount(np.asarray(y_train, int),
                             minlength=self.num_classes).astype(float)
        prior = counts / counts.sum()
        return tau * np.log(prior + eps)

    def apply_bias_and_temperature(self, proba, bias=None, temperature=1.0):
        logits = self._proba_to_logits(proba)
        if bias is not None:
            logits = logits - bias.reshape(1, -1)
        logits = logits / max(float(temperature), 1e-6)
        return self._softmax(logits)

    def select_best_calibration(self, valid_proba, y_valid, y_train,
                                objective="macro_f1", calibration_mode="two_stage"):
        """Two-stage selection: choose tau (logit adjustment) and temperature on the
        validation set. Stage 1 fixes tau by macro-F1; stage 2 fixes temperature given
        that tau. This avoids the temperature non-identifiability that arises when the
        full (tau, temperature) grid is scored jointly under a macro-F1 objective, where
        temperature is a monotone rescaling that leaves argmax — and hence macro-F1 —
        unchanged, so the joint search locks temperature to the first grid value by an
        iteration-order artifact rather than by fit."""
        y_valid = np.asarray(y_valid); y_train = np.asarray(y_train)

        def _score(cal):
            pred = cal.argmax(axis=1)
            if objective == "macro_f1":
                return f1_score(y_valid, pred, average="macro", zero_division=0)
            if objective == "nll":
                return -self.safe_nll(y_valid, cal)
            raise ValueError(objective)

        if calibration_mode == "two_stage":
            # ---- stage 1: pick tau at temperature = 1.0 by the label-aware objective
            best_tau, best_tau_score = 0.0, -1e9
            for tau in self.tau_grid:
                bias = self.logit_adjustment_bias(y_train, tau=tau)
                cal = self.apply_bias_and_temperature(valid_proba, bias=bias,
                                                      temperature=1.0)
                s = _score(cal)
                if s > best_tau_score:
                    best_tau_score, best_tau = s, tau
            # ---- stage 2: pick temperature under NLL (a proper scoring rule that IS
            #      sensitive to temperature), holding tau fixed
            bias = self.logit_adjustment_bias(y_train, tau=best_tau)
            best_temp, best_temp_score = 1.0, -1e9
            for temp in self.temperature_grid:
                cal = self.apply_bias_and_temperature(valid_proba, bias=bias,
                                                      temperature=temp)
                s = -self.safe_nll(y_valid, cal)   # stage-2 objective is NLL by design
                if s > best_temp_score:
                    best_temp_score, best_temp = s, temp
            return {"tau": best_tau, "temperature": best_temp,
                    "score": float(best_tau_score)}

        # ---- legacy single-stage joint grid (kept for comparison / ablation) ----
        best = {"tau": 0.0, "temperature": 1.0, "score": -1e9}
        for tau in self.tau_grid:
            bias = self.logit_adjustment_bias(y_train, tau=tau)
            for temp in self.temperature_grid:
                cal = self.apply_bias_and_temperature(valid_proba, bias=bias,
                                                      temperature=temp)
                s = _score(cal)
                if s > best["score"]:
                    best = {"tau": tau, "temperature": temp, "score": float(s)}
        return best

    # ---------- estimators ----------
    def train_da_minilm_mlp(self, X_tr, y_tr, w_tr, X_va, y_va, X_te, seed=42):
        set_all_seeds(seed)
        tr_ds = _EmbeddingDataset(X_tr, y_tr, w_tr)
        va_ds = _EmbeddingDataset(X_va, y_va)
        te_ds = _EmbeddingDataset(X_te, None)
        g = torch.Generator(); g.manual_seed(seed)
        tr_ld = DataLoader(tr_ds, batch_size=MLP_BATCH_SIZE, shuffle=True, generator=g)
        va_ld = DataLoader(va_ds, batch_size=MLP_BATCH_SIZE, shuffle=False)
        te_ld = DataLoader(te_ds, batch_size=MLP_BATCH_SIZE, shuffle=False)

        model = _DAMiniLMClassifier(input_dim=X_tr.shape[1],
                                    num_classes=self.num_classes).to(self.device)
        opt = torch.optim.AdamW(model.parameters(), lr=MLP_LR,
                                weight_decay=MLP_WEIGHT_DECAY)
        best_state, best_f1, bad = None, -1.0, 0
        for _ in range(MLP_EPOCHS):
            model.train()
            for b in tr_ld:
                x = b["x"].to(self.device); y = b["y"].to(self.device)
                sw = b["sample_weight"].to(self.device)
                loss = (F.cross_entropy(model(x), y, reduction="none") * sw).mean()
                opt.zero_grad(); loss.backward(); opt.step()
            va_p = self._predict_proba(model, va_ld)
            f1 = f1_score(y_va, va_p.argmax(1), average="macro", zero_division=0)
            if f1 > best_f1:
                best_f1, bad = f1, 0
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= MLP_PATIENCE:
                    break
        if best_state is not None:
            model.load_state_dict(best_state)
        return (self._predict_proba(model, va_ld),
                self._predict_proba(model, te_ld), best_f1)

    def _predict_proba(self, model, loader):
        model.eval(); out = []
        with torch.no_grad():
            for b in loader:
                out.append(F.softmax(model(b["x"].to(self.device)), dim=-1).cpu().numpy())
        return np.concatenate(out, axis=0)

    def train_logreg(self, X_tr, y_tr, w_tr, X_va, X_te, seed=42):
        clf = make_pipeline(
            StandardScaler(with_mean=True),
            # NOTE: the `multi_class` argument was removed in scikit-learn >=1.7;
            # multinomial is the default for softmax-capable solvers, so it is omitted
            # for forward/backward compatibility (behaviour is unchanged).
            LogisticRegression(max_iter=2000, class_weight=None,
                               random_state=seed, n_jobs=-1),
        )
        clf.fit(X_tr, y_tr, logisticregression__sample_weight=w_tr)
        return clf.predict_proba(X_va), clf.predict_proba(X_te)

    # ---------- metrics ----------
    @staticmethod
    def ordinal_mae(y, p):
        return float(np.mean(np.abs(np.asarray(y) - np.asarray(p))))

    @staticmethod
    def within_one_accuracy(y, p):
        return float(np.mean(np.abs(np.asarray(y) - np.asarray(p)) <= 1))

    @staticmethod
    def expected_calibration_error(y, proba, n_bins=10):
        y = np.asarray(y); proba = np.asarray(proba)
        conf = proba.max(1); pred = proba.argmax(1)
        correct = (pred == y).astype(float)
        ece = 0.0; edges = np.linspace(0, 1, n_bins + 1)
        for i in range(n_bins):
            lo, hi = edges[i], edges[i + 1]
            m = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
            if m.sum() == 0:
                continue
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
        return float(ece)

    def multiclass_brier(self, y, proba):
        y = np.asarray(y)
        oh = np.eye(self.num_classes)[y]
        return float(np.mean(np.sum((proba - oh) ** 2, axis=1)))

    def safe_nll(self, y, proba):
        return float(log_loss(y, np.clip(proba, 1e-12, 1.0),
                              labels=list(range(self.num_classes))))

    def evaluate(self, y_true, y_pred, proba=None, **meta):
        row = {
            "accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            "macro_precision": precision_score(y_true, y_pred, average="macro",
                                               zero_division=0),
            "macro_recall": recall_score(y_true, y_pred, average="macro",
                                         zero_division=0),
            "ordinal_mae": self.ordinal_mae(y_true, y_pred),
            "within_one_accuracy": self.within_one_accuracy(y_true, y_pred),
        }
        if proba is not None:
            row["ece"] = self.expected_calibration_error(y_true, proba)
            row["brier"] = self.multiclass_brier(y_true, proba)
            row["nll"] = self.safe_nll(y_true, proba)
        row.update(meta)
        return row


### 1e. `DAMiniLMExperimentRunner`

Orchestrates the recency-λ selection, calibration, and per-seed evaluation. Its public
surface matches what the collapse battery calls:

- `run_single_experiment(...)` → one `(model_kind, text_mode, split, seed)` run, returns
  `(row, lambda_df, pred, proba)`
- `run_full_battery(...)` → loops `model_kinds` × `text_modes` × `split_modes` × `seeds`,
  returns `(main_df, lambda_df, prediction_store)`
- `run_paired_bootstrap(...)` → paired bootstrap of macro-F1 differences between two
  named models across seeds

Model display names are emitted exactly as the manuscript uses them: the MLP head is
**`DA-MiniLM-RWAC`** and the reweighted logistic baseline is **`MiniLM-LogReg-RW`**. The
unweighted variant (**`Uniform-LR`**) is produced by the caller passing
`use_reweighting=False`.


In [10]:
# ============================================================
# 1e. DAMiniLMExperimentRunner  (dataset-agnostic)
# ============================================================
MODEL_DISPLAY_NAMES = {
    ("da_minilm_mlp", True):  "DA-MiniLM-RWAC",
    ("minilm_logreg", True):  "MiniLM-LogReg-RW",
    ("minilm_logreg", False): "Uniform-LR",
    ("da_minilm_mlp", False): "DA-MiniLM-Unweighted",
}


class DAMiniLMExperimentRunner:
    def __init__(self, splitter, model, num_classes, output_dir):
        self.splitter = splitter
        self.model = model
        self.num_classes = int(num_classes)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

    # ---------- one run ----------
    def run_single_experiment(self, X, y, time_rank, train_idx, valid_idx, test_idx,
                              text_mode, split_mode, seed, model_kind="da_minilm_mlp",
                              use_reweighting=True, use_calibration=True,
                              calibration_mode="two_stage"):
        m = self.model
        Xtr, Xva, Xte = X[train_idx], X[valid_idx], X[test_idx]
        ytr, yva, yte = y[train_idx], y[valid_idx], y[test_idx]
        ttr = time_rank[train_idx]
        model_name = MODEL_DISPLAY_NAMES[(model_kind, use_reweighting)]

        # recency-lambda sweep (only meaningful when reweighting; else single pass)
        lam_grid = RECENCY_LAMBDAS if use_reweighting else [0.0]
        candidates, stored = [], {}
        for lam in lam_grid:
            if use_reweighting:
                w = m.combined_sample_weights(ytr, ttr, recency_lambda=lam)
            else:
                w = np.ones(len(ytr), dtype=np.float32)  # unweighted -> Uniform-LR

            if model_kind == "da_minilm_mlp":
                va_p, te_p, raw_f1 = m.train_da_minilm_mlp(Xtr, ytr, w, Xva, yva, Xte,
                                                           seed=seed)
            elif model_kind == "minilm_logreg":
                va_p, te_p = m.train_logreg(Xtr, ytr, w, Xva, Xte, seed=seed)
                raw_f1 = f1_score(yva, va_p.argmax(1), average="macro", zero_division=0)
            else:
                raise ValueError(f"Unknown model_kind: {model_kind}")

            if use_calibration:
                cal = m.select_best_calibration(va_p, yva, ytr, objective="macro_f1",
                                                calibration_mode=calibration_mode)
            else:
                cal = {"tau": 0.0, "temperature": 1.0}
            bias = m.logit_adjustment_bias(ytr, tau=cal["tau"])
            va_cal = m.apply_bias_and_temperature(va_p, bias=bias,
                                                  temperature=cal["temperature"])
            va_f1 = f1_score(yva, va_cal.argmax(1), average="macro", zero_division=0)

            candidates.append({"recency_lambda": lam, "valid_macro_f1": va_f1,
                               "raw_valid_macro_f1": raw_f1,
                               "tau": cal["tau"], "temperature": cal["temperature"]})
            stored[lam] = (va_p, te_p, cal)

        lambda_df = pd.DataFrame(candidates).sort_values("valid_macro_f1",
                                                         ascending=False)
        best_lam = float(lambda_df.iloc[0]["recency_lambda"])
        va_p, te_p, cal = stored[best_lam]
        bias = m.logit_adjustment_bias(ytr, tau=cal["tau"])
        te_cal = m.apply_bias_and_temperature(te_p, bias=bias,
                                              temperature=cal["temperature"])
        pred = te_cal.argmax(1)

        row = m.evaluate(
            yte, pred, proba=te_cal,
            model=model_name, split_mode=split_mode, text_mode=text_mode, seed=seed,
            recency_lambda=best_lam, tau=cal["tau"], temperature=cal["temperature"],
            n_train=len(train_idx), n_valid=len(valid_idx), n_test=len(test_idx),
            selected_valid_macro_f1=float(lambda_df.iloc[0]["valid_macro_f1"]),
        )
        return row, lambda_df, pred, te_cal

    # ---------- full battery ----------
    def run_full_battery(self, df, embeddings, seeds, text_modes, split_modes,
                         model_kinds=("minilm_logreg", "da_minilm_mlp"),
                         calibration_mode="two_stage"):
        y_all = df["label_id"].values.astype(int)
        t_all = df["time_rank"].values.astype(float)
        main_rows, lambda_rows = [], []
        pred_store = {}
        for text_mode in text_modes:
            X = embeddings[text_mode]
            for split_mode in split_modes:
                for seed in seeds:
                    tr, va, te = self.splitter.get_split(df, split_mode, seed=seed)
                    for kind in model_kinds:
                        row, lam_df, pred, proba = self.run_single_experiment(
                            X, y_all, t_all, tr, va, te,
                            text_mode=text_mode, split_mode=split_mode, seed=seed,
                            model_kind=kind, use_reweighting=True,
                            use_calibration=True, calibration_mode=calibration_mode,
                        )
                        main_rows.append(row)
                        for _, r in lam_df.iterrows():
                            lambda_rows.append({"model": row["model"],
                                                "text_mode": text_mode,
                                                "split_mode": split_mode, "seed": seed,
                                                **r.to_dict()})
                        pred_store[(row["model"], text_mode, split_mode, seed)] = {
                            "y_test": y_all[te], "pred": pred, "proba": proba}
                        print(f"  [{row['model']:>16s}] text={text_mode} "
                              f"split={split_mode} seed={seed}  "
                              f"macroF1={row['macro_f1']:.4f}  "
                              f"lambda={row['recency_lambda']}")
        return (pd.DataFrame(main_rows), pd.DataFrame(lambda_rows), pred_store)

    # ---------- paired bootstrap ----------
    def run_paired_bootstrap(self, prediction_store, text_modes, split_modes, seeds,
                             model_a="DA-MiniLM-RWAC", model_b="MiniLM-LogReg-RW",
                             n_boot=2000):
        rows = []
        for text_mode in text_modes:
            for split_mode in split_modes:
                for seed in seeds:
                    ka = (model_a, text_mode, split_mode, seed)
                    kb = (model_b, text_mode, split_mode, seed)
                    if ka not in prediction_store or kb not in prediction_store:
                        continue
                    ra, rb = prediction_store[ka], prediction_store[kb]
                    y_true = np.asarray(ra["y_test"])
                    pa, pb = np.asarray(ra["pred"]), np.asarray(rb["pred"])
                    rng = np.random.RandomState(seed)
                    n = len(y_true)
                    diffs = np.empty(n_boot)
                    for b in range(n_boot):
                        idx = rng.randint(0, n, size=n)
                        diffs[b] = (
                            f1_score(y_true[idx], pa[idx], average="macro",
                                     zero_division=0)
                            - f1_score(y_true[idx], pb[idx], average="macro",
                                       zero_division=0))
                    lo, hi = np.percentile(diffs, [2.5, 97.5])
                    p = min(2.0 * min(float(np.mean(diffs <= 0)),
                                      float(np.mean(diffs >= 0))), 1.0)
                    rows.append({"text_mode": text_mode, "split_mode": split_mode,
                                 "seed": seed, "model_a": model_a, "model_b": model_b,
                                 "mean_diff": float(diffs.mean()),
                                 "ci_lower_95": float(lo), "ci_upper_95": float(hi),
                                 "p_value": p,
                                 "significant_at_95": bool(lo > 0 or hi < 0)})
        return pd.DataFrame(rows)


print("TemporalSplitter, DAMiniLMModel, DAMiniLMExperimentRunner defined.")


TemporalSplitter, DAMiniLMModel, DAMiniLMExperimentRunner defined.


In [11]:
# Sanity check: confirm the four required classes are defined before continuing.
_required = ["MultiFCPreprocessor", "TemporalSplitter", "DAMiniLMModel",
             "DAMiniLMExperimentRunner"]
_missing = [name for name in _required if name not in dir()]
assert not _missing, f"Missing class definitions: {_missing}."
print("All four project classes are defined:", ", ".join(_required))


All four project classes are defined: MultiFCPreprocessor, TemporalSplitter, DAMiniLMModel, DAMiniLMExperimentRunner


## Section 2 — Load and preprocess MultiFC (once)

In [12]:
prep = MultiFCPreprocessor(output_dir=OUTPUT_DIR)


def _resolve(path, fname):
    """Fall back to a recursive Kaggle-input search if the literal path is missing."""
    if Path(path).exists():
        return path
    for base in ["/kaggle/input", "/mnt/data", "."]:
        b = Path(base)
        if b.exists():
            hits = list(b.rglob(fname))
            if hits:
                return str(hits[0])
    raise FileNotFoundError(f"Could not locate {fname}. Edit the Section 0 paths.")


_train = _resolve(MULTIFC_TRAIN, "train.tsv")
_dev   = _resolve(MULTIFC_DEV,   "dev.tsv")
_test  = _resolve(MULTIFC_TEST,  "test.tsv") if Path(MULTIFC_TEST).name else None
print("Resolved paths:\n  train:", _train, "\n  dev  :", _dev, "\n  test :", _test)

df3 = prep.run_pipeline(_train, _dev, _test)

# save the label-harmonisation audit required by the manuscript
prep.label_harmonization_report(
    save_path=f"{OUTPUT_DIR}/multifc_label_harmonization_report.csv")

# Canonical dedup (keep_first) then re-establish chronological order + time_rank.
try:
    _audit = prep.audit_duplicate_statements(df3)
    df3 = prep.drop_duplicate_statements(df3, policy="keep_first", audit_df=_audit)
    df3 = prep.parse_and_sort_by_date(df3)
    df3 = prep.add_time_rank(df3)
    print("Applied dedup(keep_first) + chronological re-sort + time_rank.")
except Exception as e:
    print("NOTE: dedup step skipped or unavailable ->", repr(e))

df3 = df3.reset_index(drop=True)

assert "label_id" in df3.columns and "time_rank" in df3.columns
assert f"model_text__{TEXT_MODE}" in df3.columns, (
    f"Expected column model_text__{TEXT_MODE} not found. Available: "
    f"{[c for c in df3.columns if c.startswith('model_text__')]}")
print(f"\nFinal MultiFC rows: {len(df3)}")
print("3-class label_id distribution:")
print(df3["label_id"].value_counts().sort_index()
      .rename(index={i: n for i, n in enumerate(THREECLASS_ORDER)}))


Resolved paths:
  train: /kaggle/input/datasets/badalahmmed/multifc/train.tsv 
  dev  : /kaggle/input/datasets/badalahmmed/multifc/dev.tsv 
  test : /kaggle/input/datasets/badalahmmed/multifc/test.tsv
MultiFC pipeline complete.
  labelled rows kept : 27192  (dropped 4241 unmappable/empty of 31433, 13.5%)
  official UNLABELLED test rows ignored (not scorable): 3491
  3-class distribution: {'false': 14140, 'true': 7836, 'mixed': 5216}
Saved label-harmonisation report: /kaggle/working/multifc_label_collapse_results/multifc_label_harmonization_report.csv
Duplicate-claim audit: 59 claim strings occur >1x (60 redundant rows).
Dropped 60 duplicate-claim rows (policy=keep_first). Remaining: 27132.
Applied dedup(keep_first) + chronological re-sort + time_rank.

Final MultiFC rows: 27132
3-class label_id distribution:
label_id
false    14107
mixed     5200
true      7825
Name: count, dtype: int64


## Section 3 — Compute frozen MiniLM embeddings (once, reused everywhere)

In [13]:
# A shared model instance ONLY for encoding (num_classes irrelevant for encode).
_encoder = DAMiniLMModel(output_dir=OUTPUT_DIR, num_classes=3)

_text_modes_to_embed = [TEXT_MODE] + (["claim_only"] if ALSO_RUN_CLAIM_ONLY else [])

embeddings = {}
for _mode in _text_modes_to_embed:
    _col = f"model_text__{_mode}"
    print(f"Encoding text mode: {_mode}  ({_col})")
    embeddings[_mode] = _encoder.encode_texts_minilm(
        df3[_col].tolist(), cache_key=f"multifc_collapse_{_mode}")
    print(f"  -> embedding shape: {embeddings[_mode].shape}")

TEXT_MODES = list(embeddings.keys())
print("\nEmbeddings ready for text modes:", TEXT_MODES)


Encoding text mode: claim_plus_safe_metadata  (model_text__claim_plus_safe_metadata)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/212 [00:00<?, ?it/s]

Saved embeddings: /kaggle/working/multifc_label_collapse_results/embeddings_multifc_collapse_claim_plus_safe_metadata.npy (27132, 384)
  -> embedding shape: (27132, 384)

Embeddings ready for text modes: ['claim_plus_safe_metadata']


## Section 4 — Run the collapse battery

For each granularity: build a fresh model + runner (correct `num_classes`, recomputed
class weights), reuse the cached embeddings, and run DA-MiniLM-RWAC + MiniLM-LogReg-RW
via the standard battery, then Uniform-LR (unweighted LogReg) as an extra pass, then the
paired bootstrap DA-MiniLM-RWAC − MiniLM-LogReg-RW.


In [14]:
def build_runner(num_classes):
    """Fresh model + runner configured for a given class count, reusing the shared
    embedding cache so encoding never repeats."""
    model = DAMiniLMModel(output_dir=f"{OUTPUT_DIR}/K{num_classes}",
                          num_classes=num_classes)
    model._embedding_cache = _encoder._embedding_cache
    splitter = TemporalSplitter(train_ratio=0.70, valid_ratio=0.15, test_ratio=0.15)
    runner = DAMiniLMExperimentRunner(splitter=splitter, model=model,
                                      num_classes=num_classes,
                                      output_dir=f"{OUTPUT_DIR}/K{num_classes}")
    return model, splitter, runner


all_main_rows, all_bootstrap_rows = [], []
prediction_stores = {}

for scheme in SCHEMES:
    cfg = COLLAPSE_SCHEMES[scheme]
    K = cfg["num_classes"]
    print("\n" + "=" * 72)
    print(f"SCHEME = {scheme}   (num_classes = {K})")
    print("=" * 72)

    collapser = LabelCollapser(scheme)
    df_c = collapser.transform(df3).reset_index(drop=True)
    print(collapser.describe_collapse(df_c).to_string(index=False))

    os.makedirs(f"{OUTPUT_DIR}/K{K}", exist_ok=True)
    model, splitter, runner = build_runner(K)

    # ---- DA-MiniLM-RWAC + MiniLM-LogReg-RW via the standard battery ----
    main_df, lambda_df, pred_store = runner.run_full_battery(
        df=df_c, embeddings=embeddings, seeds=SEEDS,
        text_modes=TEXT_MODES, split_modes=[SPLIT_MODE],
        model_kinds=("minilm_logreg", "da_minilm_mlp"),
        calibration_mode="two_stage",
    )

    # ---- Uniform-LR (unweighted frozen-embedding LogReg) ----
    y_all = df_c["label_id"].values.astype(int)
    t_all = df_c["time_rank"].values.astype(float)
    for _mode in TEXT_MODES:
        X = embeddings[_mode]
        for seed in SEEDS:
            tr, va, te = splitter.get_split(df_c, SPLIT_MODE, seed=seed)
            row_u, _, pred_u, proba_u = runner.run_single_experiment(
                X, y_all, t_all, tr, va, te,
                text_mode=_mode, split_mode=SPLIT_MODE, seed=seed,
                model_kind="minilm_logreg", use_reweighting=False,
                use_calibration=True, calibration_mode="two_stage",
            )
            main_df = pd.concat([main_df, pd.DataFrame([row_u])], ignore_index=True)
            pred_store[(row_u["model"], _mode, SPLIT_MODE, seed)] = {
                "y_test": y_all[te], "pred": pred_u, "proba": proba_u}

    main_df["scheme"] = scheme
    main_df["num_classes"] = K
    all_main_rows.append(main_df)
    prediction_stores[scheme] = pred_store

    # ---- Paired bootstrap: DA-MiniLM-RWAC minus MiniLM-LogReg-RW ----
    boot_df = runner.run_paired_bootstrap(
        pred_store, text_modes=TEXT_MODES, split_modes=[SPLIT_MODE], seeds=SEEDS,
        model_a="DA-MiniLM-RWAC", model_b="MiniLM-LogReg-RW", n_boot=N_BOOT)
    if len(boot_df):
        boot_df["scheme"] = scheme
        boot_df["num_classes"] = K
        all_bootstrap_rows.append(boot_df)

main_results = pd.concat(all_main_rows, ignore_index=True)
bootstrap_results = (pd.concat(all_bootstrap_rows, ignore_index=True)
                     if all_bootstrap_rows else pd.DataFrame())

main_results.to_csv(f"{OUTPUT_DIR}/collapse_main_results.csv", index=False)
if len(bootstrap_results):
    bootstrap_results.to_csv(f"{OUTPUT_DIR}/collapse_bootstrap_results.csv", index=False)

print("\nAll schemes complete.")
print("Main results rows:", len(main_results))



SCHEME = 3class   (num_classes = 3)
 three_id three_label  collapsed_id collapsed_label  n_rows  collapsed_total
        0       false             0           false   14107            14107
        1       mixed             1           mixed    5200             5200
        2        true             2            true    7825             7825
  [MiniLM-LogReg-RW] text=claim_plus_safe_metadata split=chronological seed=0  macroF1=0.4500  lambda=0.2
  [  DA-MiniLM-RWAC] text=claim_plus_safe_metadata split=chronological seed=0  macroF1=0.4497  lambda=0.2
  [MiniLM-LogReg-RW] text=claim_plus_safe_metadata split=chronological seed=1  macroF1=0.4500  lambda=0.2
  [  DA-MiniLM-RWAC] text=claim_plus_safe_metadata split=chronological seed=1  macroF1=0.4518  lambda=0.2
  [MiniLM-LogReg-RW] text=claim_plus_safe_metadata split=chronological seed=2  macroF1=0.4500  lambda=0.2
  [  DA-MiniLM-RWAC] text=claim_plus_safe_metadata split=chronological seed=2  macroF1=0.4514  lambda=0.2
  [MiniLM-LogReg-RW

## Section 5 — Summary tables

In [15]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

summary = (main_results
    .groupby(["scheme", "num_classes", "model", "text_mode"])
    .agg(macro_f1_mean=("macro_f1", "mean"),
         macro_f1_std =("macro_f1", "std"),
         accuracy_mean=("accuracy", "mean"),
         macro_recall_mean=("macro_recall", "mean"),
         ece_mean=("ece", "mean"),
         n_runs=("macro_f1", "count"))
    .reset_index())

_scheme_rank = {"3class": 0, "2class": 1}
summary["_r"] = summary["scheme"].map(_scheme_rank)
summary = summary.sort_values(["_r", "text_mode", "macro_f1_mean"],
                              ascending=[True, True, False]).drop(columns="_r")

print("=" * 90)
print("PER-MODEL SUMMARY  (macro-F1 mean +/- std across seeds)")
print("=" * 90)
print(summary.to_string(index=False))
summary.to_csv(f"{OUTPUT_DIR}/collapse_summary_by_model.csv", index=False)


PER-MODEL SUMMARY  (macro-F1 mean +/- std across seeds)
scheme  num_classes            model                text_mode  macro_f1_mean  macro_f1_std  accuracy_mean  macro_recall_mean  ece_mean  n_runs
3class            3   DA-MiniLM-RWAC claim_plus_safe_metadata       0.451368      0.001501       0.556462           0.455204  0.034280       5
3class            3 MiniLM-LogReg-RW claim_plus_safe_metadata       0.450034      0.000000       0.547912           0.457411  0.026799       5
3class            3       Uniform-LR claim_plus_safe_metadata       0.447565      0.000000       0.544226           0.453456  0.024007       5
2class            2       Uniform-LR claim_plus_safe_metadata       0.598533      0.000000       0.719165           0.607119  0.015357       5
2class            2 MiniLM-LogReg-RW claim_plus_safe_metadata       0.598132      0.000000       0.727027           0.602718  0.010853       5
2class            2   DA-MiniLM-RWAC claim_plus_safe_metadata       0.595658      0.01

In [16]:
# ---- Headline effect-size table: DA-MiniLM-RWAC minus MiniLM-LogReg-RW ----
def effect_table(text_mode):
    rows = []
    for scheme in SCHEMES:
        K = COLLAPSE_SCHEMES[scheme]["num_classes"]
        sub = summary[(summary.scheme == scheme) & (summary.text_mode == text_mode)]
        try:
            f_rwac = sub.loc[sub.model == "DA-MiniLM-RWAC", "macro_f1_mean"].iloc[0]
            f_rw   = sub.loc[sub.model == "MiniLM-LogReg-RW", "macro_f1_mean"].iloc[0]
        except IndexError:
            continue
        abs_diff = f_rwac - f_rw
        rel = 100.0 * abs_diff / f_rw if f_rw else np.nan
        ci_l = ci_u = p = np.nan
        sig = None
        if len(bootstrap_results):
            b = bootstrap_results[(bootstrap_results.scheme == scheme) &
                                  (bootstrap_results.text_mode == text_mode)]
            if len(b):
                ci_l = b["ci_lower_95"].mean(); ci_u = b["ci_upper_95"].mean()
                p = b["p_value"].mean()
                sig = bool((b["significant_at_95"]).mean() >= 0.5)
        rows.append({"scheme": scheme, "K": K,
                     "RWAC_macroF1": round(f_rwac, 4),
                     "LogRegRW_macroF1": round(f_rw, 4),
                     "abs_diff": round(abs_diff, 4),
                     "rel_pct": round(rel, 2) if rel == rel else np.nan,
                     "boot_CI95_low": round(ci_l, 4) if ci_l == ci_l else np.nan,
                     "boot_CI95_high": round(ci_u, 4) if ci_u == ci_u else np.nan,
                     "boot_p": round(p, 4) if p == p else np.nan,
                     "sig_at_95": sig})
    return pd.DataFrame(rows)

for _mode in TEXT_MODES:
    print("=" * 90)
    print(f"HEADLINE EFFECT SIZE  DA-MiniLM-RWAC  minus  MiniLM-LogReg-RW   "
          f"[text_mode={_mode}]")
    print("(absolute macro-F1 difference and 95% bootstrap CI reported before "
          "significance)")
    print("=" * 90)
    et = effect_table(_mode)
    print(et.to_string(index=False))
    et.to_csv(f"{OUTPUT_DIR}/collapse_effect_size_{_mode}.csv", index=False)
    print()


HEADLINE EFFECT SIZE  DA-MiniLM-RWAC  minus  MiniLM-LogReg-RW   [text_mode=claim_plus_safe_metadata]
(absolute macro-F1 difference and 95% bootstrap CI reported before significance)
scheme  K  RWAC_macroF1  LogRegRW_macroF1  abs_diff  rel_pct  boot_CI95_low  boot_CI95_high  boot_p  sig_at_95
3class  3        0.4514            0.4500    0.0013     0.30        -0.0141          0.0172  0.8446      False
2class  2        0.5957            0.5981   -0.0025    -0.41        -0.0194          0.0140  0.3066      False



## Section 5b — Seed-level effect-size summary

Between-seed variability of the DA-MiniLM-RWAC − MiniLM-LogReg-RW macro-F1 difference:
effect size + paired *t*-interval lead, sign test corroborates.


In [17]:
from scipy import stats

def seed_level_effect(main_results, model_a="DA-MiniLM-RWAC",
                      model_b="MiniLM-LogReg-RW",
                      text_mode=TEXT_MODE, split_mode=SPLIT_MODE):
    rows = []
    for scheme in SCHEMES:
        sub = main_results[(main_results.scheme == scheme) &
                           (main_results.text_mode == text_mode) &
                           (main_results.split_mode == split_mode)]
        a = sub[sub.model == model_a].sort_values("seed")["macro_f1"].values
        b = sub[sub.model == model_b].sort_values("seed")["macro_f1"].values
        if len(a) == 0 or len(b) == 0:
            continue
        n = min(len(a), len(b))
        diffs = a[:n] - b[:n]
        mean = diffs.mean(); sd = diffs.std(ddof=1) if n > 1 else 0.0
        se = sd / np.sqrt(n) if n > 1 else 0.0
        tcrit = stats.t.ppf(0.975, n - 1) if n > 1 else 0.0
        ci_l, ci_u = mean - tcrit * se, mean + tcrit * se
        t_stat, p_t = stats.ttest_1samp(diffs, 0.0) if n > 1 else (np.nan, np.nan)
        n_pos = int((diffs > 0).sum())
        p_sign = stats.binomtest(n_pos, n, 0.5).pvalue
        rows.append({"scheme": scheme, "K": COLLAPSE_SCHEMES[scheme]["num_classes"],
                     "n_seeds": n, "mean_diff": round(mean, 4),
                     "between_seed_sd": round(sd, 4),
                     "t_CI95_low": round(ci_l, 4), "t_CI95_high": round(ci_u, 4),
                     "t_p": round(p_t, 4) if p_t == p_t else np.nan,
                     "seeds_favoring_A": f"{n_pos}/{n}",
                     "sign_p": round(p_sign, 4)})
    df = pd.DataFrame(rows)
    _r = {"3class": 0, "2class": 1}
    return df.assign(_r=df["scheme"].map(_r)).sort_values("_r").drop(columns="_r")

seed_effect = seed_level_effect(main_results)
print("=" * 96)
print(f"SEED-LEVEL EFFECT SIZE  DA-MiniLM-RWAC minus MiniLM-LogReg-RW  "
      f"[text_mode={TEXT_MODE}]")
print("=" * 96)
print(seed_effect.to_string(index=False))
seed_effect.to_csv(f"{OUTPUT_DIR}/collapse_seed_level_effect.csv", index=False)

print("""
Reading guide
-------------
- mean_diff + t_CI95: primary. If the interval excludes 0, the advantage is
  detectable at that granularity across seeds.
- seeds_favoring_A: corroboration. At n=5 the sign test's p-value floor is 0.0625,
  so it never reaches 0.05 here by design.
- Agreement between this between-seed interval and the Section 5 within-test bootstrap
  is the robust external-replication claim.
""")


SEED-LEVEL EFFECT SIZE  DA-MiniLM-RWAC minus MiniLM-LogReg-RW  [text_mode=claim_plus_safe_metadata]
scheme  K  n_seeds  mean_diff  between_seed_sd  t_CI95_low  t_CI95_high    t_p seeds_favoring_A  sign_p
3class  3        5     0.0013           0.0015     -0.0005       0.0032 0.1178              4/5   0.375
2class  2        5    -0.0025           0.0113     -0.0165       0.0115 0.6492              2/5   1.000

Reading guide
-------------
- mean_diff + t_CI95: primary. If the interval excludes 0, the advantage is
  detectable at that granularity across seeds.
- seeds_favoring_A: corroboration. At n=5 the sign test's p-value floor is 0.0625,
  so it never reaches 0.05 here by design.
- Agreement between this between-seed interval and the Section 5 within-test bootstrap
  is the robust external-replication claim.



## Section 6 — Per-class recall at each granularity

In [18]:
from sklearn.metrics import recall_score as _rs

def per_class_recall_rows():
    rows = []
    for scheme in SCHEMES:
        order = COLLAPSE_SCHEMES[scheme]["order"]
        store = prediction_stores[scheme]
        for (model_name, text_mode, split_mode, seed), rec in store.items():
            if text_mode not in TEXT_MODES:
                continue
            r = _rs(rec["y_test"], rec["pred"], average=None,
                    labels=list(range(len(order))), zero_division=0)
            for cid, cname in enumerate(order):
                rows.append({"scheme": scheme, "model": model_name,
                             "text_mode": text_mode, "seed": seed,
                             "class": cname, "recall": r[cid]})
    return pd.DataFrame(rows)

pcr = per_class_recall_rows()
pcr_summary = (pcr.groupby(["scheme", "model", "text_mode", "class"])
                  .agg(recall_mean=("recall", "mean"),
                       recall_std=("recall", "std"))
                  .reset_index())

for _mode in TEXT_MODES:
    print("=" * 90)
    print(f"PER-CLASS RECALL (mean over seeds)   [text_mode={_mode}]")
    print("=" * 90)
    piv = (pcr_summary[pcr_summary.text_mode == _mode]
           .pivot_table(index=["scheme", "model"], columns="class",
                        values="recall_mean"))
    print(piv.to_string())
    print()

pcr_summary.to_csv(f"{OUTPUT_DIR}/collapse_per_class_recall.csv", index=False)


PER-CLASS RECALL (mean over seeds)   [text_mode=claim_plus_safe_metadata]
class                       false     mixed      true
scheme model                                         
2class DA-MiniLM-RWAC    0.770705       NaN  0.451319
       MiniLM-LogReg-RW  0.813350       NaN  0.392086
       Uniform-LR        0.796972       NaN  0.417266
3class DA-MiniLM-RWAC    0.684169  0.278805  0.402638
       MiniLM-LogReg-RW  0.670746  0.412518  0.288969
       Uniform-LR        0.665219  0.379801  0.315348



## Section 6b — Chance-level baselines per granularity

Macro-F1 chance is **not** 1/K. Reporting the stratified-random and majority-class
floors, plus each model's margin over the stratified floor, makes macro-F1 comparable
across K (raw macro-F1 is not).


In [19]:
from sklearn.metrics import f1_score as _f1

def compute_chance_floors(labels_by_scheme, n_sim=2000, seed=0):
    rows = []
    for scheme, y in labels_by_scheme.items():
        y = np.asarray(y).astype(int)
        K = int(y.max()) + 1
        counts = np.bincount(y, minlength=K).astype(float)
        p = counts / counts.sum()
        rng = np.random.RandomState(seed)
        maj = np.full(len(y), int(p.argmax()))
        maj_acc = float((maj == y).mean())
        maj_f1 = _f1(y, maj, average="macro", zero_division=0)
        strat_f1 = np.empty(n_sim); unif_f1 = np.empty(n_sim)
        for b in range(n_sim):
            strat_f1[b] = _f1(y, rng.choice(K, size=len(y), p=p),
                              average="macro", zero_division=0)
            unif_f1[b]  = _f1(y, rng.randint(0, K, size=len(y)),
                              average="macro", zero_division=0)
        rows.append({"scheme": scheme, "K": K, "N_test": int(counts.sum()),
                     "majority_prevalence": round(p.max(), 4),
                     "uniform_acc_1overK": round(1.0 / K, 4),
                     "uniform_macroF1": round(unif_f1.mean(), 4),
                     "stratified_acc": round(float((p ** 2).sum()), 4),
                     "stratified_macroF1": round(strat_f1.mean(), 4),
                     "stratified_macroF1_p95": round(np.percentile(strat_f1, 95), 4),
                     "majority_acc": round(maj_acc, 4),
                     "majority_macroF1": round(maj_f1, 4)})
    return pd.DataFrame(rows)

_splitter_ref = TemporalSplitter(train_ratio=0.70, valid_ratio=0.15, test_ratio=0.15)
_test_labels_by_scheme = {}
for scheme in SCHEMES:
    df_c = LabelCollapser(scheme).transform(df3).reset_index(drop=True)
    _tr, _va, _te = _splitter_ref.get_split(df_c, SPLIT_MODE, seed=SEEDS[0])
    _test_labels_by_scheme[scheme] = df_c["label_id"].values[_te]

floors = compute_chance_floors(_test_labels_by_scheme, n_sim=2000, seed=0)

_scheme_rank = {"3class": 0, "2class": 1}
obs = (main_results.groupby(["scheme", "model"])["macro_f1"].mean().reset_index())
margin_rows = []
for _, fr in floors.iterrows():
    s = fr["scheme"]; floor = fr["stratified_macroF1"]
    for model in ["DA-MiniLM-RWAC", "MiniLM-LogReg-RW", "Uniform-LR"]:
        m = obs[(obs.scheme == s) & (obs.model == model)]
        if len(m):
            mf = float(m["macro_f1"].iloc[0])
            margin_rows.append({"scheme": s, "K": int(fr["K"]), "model": model,
                                "macro_f1": round(mf, 4),
                                "stratified_floor": floor,
                                "margin_over_chance": round(mf - floor, 4)})
margin_df = pd.DataFrame(margin_rows)
margin_df["_r"] = margin_df["scheme"].map(_scheme_rank)
margin_df = margin_df.sort_values(["_r", "macro_f1"],
                                  ascending=[True, False]).drop(columns="_r")

pd.set_option("display.width", 200); pd.set_option("display.max_columns", None)
print("=" * 96)
print("CHANCE-LEVEL FLOORS PER GRANULARITY (macro-F1 chance is NOT 1/K)")
print("=" * 96)
floors["_r"] = floors["scheme"].map(_scheme_rank)
print(floors.sort_values("_r").drop(columns="_r").to_string(index=False))

print("\n" + "=" * 96)
print("MARGIN OVER CHANCE  (raw macro-F1 is NOT comparable across K)")
print("=" * 96)
print(margin_df.to_string(index=False))

floors.drop(columns="_r").to_csv(f"{OUTPUT_DIR}/collapse_chance_floors.csv", index=False)
margin_df.to_csv(f"{OUTPUT_DIR}/collapse_margin_over_chance.csv", index=False)


CHANCE-LEVEL FLOORS PER GRANULARITY (macro-F1 chance is NOT 1/K)
scheme  K  N_test  majority_prevalence  uniform_acc_1overK  uniform_macroF1  stratified_acc  stratified_macroF1  stratified_macroF1_p95  majority_acc  majority_macroF1
3class  3    4070               0.6224              0.3333           0.3049          0.4592              0.3333                  0.3459        0.6224            0.2557
2class  2    4070               0.7951              0.5000           0.4523          0.6742              0.5001                  0.5129        0.7951            0.4429

MARGIN OVER CHANCE  (raw macro-F1 is NOT comparable across K)
scheme  K            model  macro_f1  stratified_floor  margin_over_chance
3class  3   DA-MiniLM-RWAC    0.4514            0.3333              0.1181
3class  3 MiniLM-LogReg-RW    0.4500            0.3333              0.1167
3class  3       Uniform-LR    0.4476            0.3333              0.1143
2class  2       Uniform-LR    0.5985            0.5001              

## Section 7 — Saved artifacts

In [20]:
print("Files written to", OUTPUT_DIR, ":")
for f in sorted(glob.glob(f"{OUTPUT_DIR}/**/*.csv", recursive=True)):
    print("  ", f)

print("""
Artifact guide
--------------
multifc_label_harmonization_report.csv  Raw MultiFC verdict -> 3-class mapping audit
                                        (26-site taxonomy; the manuscript documentation).
collapse_main_results.csv               Every (scheme x model x seed) run, full metrics.
collapse_summary_by_model.csv           Macro-F1 mean +/- std per scheme/model (Sec 5).
collapse_effect_size_*.csv              DA-MiniLM-RWAC minus MiniLM-LogReg-RW, abs diff +
                                        95% bootstrap CI per granularity (headline).
collapse_bootstrap_results.csv          Per-seed paired-bootstrap detail.
collapse_seed_level_effect.csv          Between-seed effect size + t-interval (Sec 5b).
collapse_per_class_recall.csv           Per-class recall per scheme/model (minority
                                        collapse diagnostic).
collapse_chance_floors.csv              Stratified/uniform/majority macro-F1 floors.
collapse_margin_over_chance.csv         Each model's margin over the stratified floor.
""")


Files written to /kaggle/working/multifc_label_collapse_results :
   /kaggle/working/multifc_label_collapse_results/collapse_bootstrap_results.csv
   /kaggle/working/multifc_label_collapse_results/collapse_chance_floors.csv
   /kaggle/working/multifc_label_collapse_results/collapse_effect_size_claim_plus_safe_metadata.csv
   /kaggle/working/multifc_label_collapse_results/collapse_main_results.csv
   /kaggle/working/multifc_label_collapse_results/collapse_margin_over_chance.csv
   /kaggle/working/multifc_label_collapse_results/collapse_per_class_recall.csv
   /kaggle/working/multifc_label_collapse_results/collapse_seed_level_effect.csv
   /kaggle/working/multifc_label_collapse_results/collapse_summary_by_model.csv
   /kaggle/working/multifc_label_collapse_results/multifc_label_harmonization_report.csv

Artifact guide
--------------
multifc_label_harmonization_report.csv  Raw MultiFC verdict -> 3-class mapping audit
                                        (26-site taxonomy; the manuscrip

## Section 8 — Reporting notes for the manuscript

- **MultiFC is external-dataset replication with independent retraining**, never
  zero-shot transfer or a generalization claim. The full DA-MiniLM-RWAC procedure is
  retrained from scratch on MultiFC embeddings.
- **Comparability with LIAR2.** The 3-class `{false, mixed, true}` taxonomy here is the
  *same* one LIAR2 is collapsed to, so the 3-class numbers are directly comparable across
  datasets. Report absolute macro-F1 and its bootstrap CI **before** any significance
  label.
- **Headline reading.** If the DA-MiniLM-RWAC − MiniLM-LogReg-RW effect that was positive
  on LIAR2 shrinks toward zero (or reverses) on MultiFC, and the simpler comparator stays
  equally/more competitive, that non-reproduction is the reportable external-replication
  finding — state it plainly rather than softening it.
- **Label harmonisation.** `multifc_label_harmonization_report.csv` documents the 26-site
  verdict → 3-class collapse; report the fraction of rows dropped as unmappable (printed
  by `run_pipeline`) so the harmonisation is auditable.
- **Leakage control.** The restricted-metadata text mode concatenates only
  `{category, speaker, checker, tags, entities}`; the fact-check verdict/evidence/title
  are excluded, matching the LIAR2 leakage discipline (justification and speaker
  credibility-count columns dropped there).
- **Chance floors.** Because MultiFC is severely imbalanced, always report macro-F1
  against the stratified floor (Section 6b), not against 1/K.
- **Test split.** MultiFC's official test set is unlabelled, so evaluation uses an
  internal chronological split of the pooled labelled rows; state this explicitly.
